# 01.1 — Rainfall Threshold & Occurrence Analysis

## 1. Mục tiêu & Ranh giới Kiến trúc Dự án

### 1.1. Target Chính Của Toàn Bộ Pipeline Forecasting
- **Target chính duy nhất của project là lượng mưa hàng ngày liên tục**: $\mathbf{Y_t \in [0, \infty)\text{ (mm)}}$.
- Tất cả các họ mô hình chính (Statistical ARIMA/SARIMA, ML Direct XGBoost/LightGBM, Tweedie GLM, Neural Forecasters) đều dự báo trực tiếp lượng mưa liên tục $Y_t$, hoàn toàn độc lập với rainfall threshold.
- Lượng mưa threshold $\tau$ **không biến target chính thành binary** và **không phải là hyperparameter chi phối toàn bộ hệ thống dự báo**.

$$\text{Main Target: } Y_t = \text{Daily Rainfall Amount (mm)}$$
$$\text{Canonical Wet Day: } \mathrm{Wet}_t = \mathbb{I}(Y_t \ge 1.0\text{ mm}) \quad \Longleftrightarrow \quad \mathrm{Dry}_t = \mathbb{I}(Y_t < 1.0\text{ mm})$$

### 1.2. Thống nhất Một Canonical Wet-Day Semantics ($Y_t \ge 1.0\text{ mm}$)
Dự án áp dụng **một định nghĩa nhất quán duy nhất** cho toàn bộ project:
$$\boxed{Y_t \ge 1.0\text{ mm/day} \implies \text{Wet Day}} \quad \Longleftrightarrow \quad \boxed{Y_t < 1.0\text{ mm/day} \implies \text{Dry Day}}$$
theo chuẩn quốc tế WMO / ETCCDI RR1 (lượng mưa $\ge 1.0\text{ mm/ngày}$ định nghĩa ngày mưa).
Định nghĩa này được dùng đồng bộ xuyên suốt ở:
- `01.1_threshold_occurrence` (khảo sát ngưỡng xuất hiện)
- `01.2_distribution_intermittency` (chẩn đoán tính gián đoạn)
- Hurdle Stage 1 Occurrence & Stage 2 Conditional Amount
- Rain-day auxiliary diagnostics ($\text{RainDay-MAE}_{\tau=1.0}$)

Không tách hai convention riêng biệt (`>= 1.0` ở climate vs `> 1.0` ở Hurdle) để đảm bảo kiến trúc nhất quán.

### 1.3. Thu hẹp vai trò của Rainfall Threshold $\tau$ xuống 2 Mục đích Phụ trợ:
1. **EDA & Thống kê mô tả (`01.1_threshold_occurrence.ipynb`)**: Khảo sát độ nhạy tỷ lệ wet/dry, vùng mưa cận zero (*near-zero / very-light precipitation regime*), và phân phối cường độ có điều kiện.
2. **Mô hình Hurdle / Two-Stage**: Định nghĩa thành phần xuất hiện (Stage 1 Occurrence) và quần thể ước lượng lượng mưa có điều kiện (Stage 2 Conditional Amount).

### 1.4. Ranh giới với Feature Engineering & Evaluation Metrics:
- **Feature Engineering F0–F3 không chứa threshold-derived features**: Không thêm `is_wet`, `wet_lag_1`, `wet_days_last_7` vào baseline F0–F3 để đảm bảo đo lường đúng độ phức tạp đặc trưng $F_0 \to F_1 \to F_2 \to F_3$.
- **Metric chính giữ độc lập với threshold**: $MAE, RMSE, CR\text{-}MAE@7, CR\text{-}Bias@7$ được tính trên toàn bộ phân phối dự báo. Metric $\text{RainDay-MAE}_\tau$ trên ngày mưa chỉ đóng vai trò **secondary diagnostic** (chẩn đoán thứ cấp).

### 1.5. Phân định Rạch ròi 2 Loại Ngưỡng:
- **Rainfall threshold ($\tau$)**: Định nghĩa sự kiện mưa vật lý (Candidate grid $[0.0, 0.1, 0.5, 1.0, 2.0, 5.0]\text{ mm}$; Canonical wet-day reference: $\tau = 1.0\text{ mm/day}$).
- **Classification probability threshold ($c$)**: Ngưỡng xác suất phân loại Stage 1 ($P(\mathrm{Wet} \mid X) > c$), được tối ưu độc lập trong quá trình validation của Hurdle model.

> **Nguyên tắc EDA:** Khảo sát độ nhạy thực nghiệm trên tập Dev Train (Zero Leakage). Tuyệt đối **không tự động chọn hay freeze threshold trong EDA**.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.evaluation.protocols import load_canonical_data
from src.eda.pipeline import compare_rain_thresholds

# 1. Nạp dữ liệu Dev Train chuẩn hóa (Zero Leakage: Dev Train 2001-01-01 -> 2018-04-30, N = 6,329)
# Validation (2018-05-01 -> 2020-04-30, N = 731) và Final Test (2020-05-01 -> 2026-04-30, N = 2,191) hoàn toàn untouched.
dev_train, val_set, test_set = load_canonical_data(return_dev_train=True)
train_df = dev_train
print(f"📊 Tập Dev Train phân tích: {len(train_df):,} ngày ({train_df['Ngày'].min().date()} → {train_df['Ngày'].max().date()})")

# 2. Bảng so sánh thực nghiệm theo danh sách candidate thresholds (mặc định inclusive_wet=True: Wet if Y >= tau)
candidate_thresholds = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]
threshold_comparison = compare_rain_thresholds(
    train_df,
    target_col='Lượng mưa',
    candidate_thresholds=candidate_thresholds,
    inclusive_wet=True
)

print("\n📋 BẢNG SO SÁNH PHÂN BỐ THỰC NGHIỆM THEO CÁC NGƯỠNG MƯA ỨNG VIÊN (Wet: Y >= tau vs Dry: Y < tau):")
print(threshold_comparison.to_string(index=False))


📊 Tập Dev Train phân tích: 6,329 ngày (2001-01-01 → 2018-04-30)

📋 BẢNG SO SÁNH PHÂN BỐ THỰC NGHIỆM THEO CÁC NGƯỠNG MƯA ỨNG VIÊN (Wet: Y >= tau vs Dry: Y < tau):
 threshold_mm no_rain_condition  no_rain_days  rain_days  no_rain_pct  rain_pct  exact_threshold_days  incremental_gap_pct  rain_mean_conditional  rain_median_conditional
          0.0         == 0.0 mm          1729       4600        27.32     72.68                  1729                 0.00                   4.94                     1.77
          0.1          < 0.1 mm          2472       3857        39.06     60.94                    46                11.74                   5.88                     2.73
          0.5          < 0.5 mm          3162       3167        49.96     50.04                    10                10.90                   7.11                     3.89
          1.0          < 1.0 mm          3582       2747        56.60     43.40                     5                 6.64                   8.09         

In [2]:
# 3. Phân tích Chi tiết Sự kiện Mưa (Rain Events & Episodes) và Boundary Censoring Robustness
target = train_df['Lượng mưa'].dropna()
total_days = len(target)

event_records = []
for th in candidate_thresholds:
    mask = (target >= th) if th > 0.0 else (target > 0.0)
    
    # Xác định các đợt mưa (contiguous wet runs)
    runs = []
    cur_len = 0
    for val in mask:
        if val:
            cur_len += 1
        else:
            if cur_len > 0:
                runs.append(cur_len)
                cur_len = 0
    right_censored = False
    if cur_len > 0:
        runs.append(cur_len)
        right_censored = True
    
    n_episodes = len(runs)
    n_rc = 1 if right_censored else 0
    mean_all = float(np.mean(runs)) if runs else 0.0
    mean_unc = float(np.mean(runs[:-1])) if (right_censored and len(runs) > 1) else mean_all
    exact_count = int((target == th).sum())
    
    event_records.append({
        "threshold_mm": th,
        "wet_days": int(mask.sum()),
        "wet_pct": round(float(mask.mean()) * 100.0, 2),
        "n_episodes": n_episodes,
        "n_right_censored": n_rc,
        "mean_duration_all": round(mean_all, 2),
        "mean_duration_uncensored": round(mean_unc, 2),
        "exact_threshold_days": exact_count
    })

df_events = pd.DataFrame(event_records)
print("\n🌧️ THỐNG KÊ ĐỢT MƯA (EPISODES) & BOUNDARY CENSORING (Observed duration within Dev Train 2001-2018):")
print(df_events.to_string(index=False))

ev_10 = df_events.loc[df_events['threshold_mm'] == 1.0].iloc[0]
ev_01 = df_events.loc[df_events['threshold_mm'] == 0.1].iloc[0]
print("\n💡 Ghi chú Boundary Censoring (Dev Train):")
print(f"   - Tại tau = 1.0 mm: {int(ev_10['n_episodes'])} đợt mưa, {int(ev_10['n_right_censored'])} right-censored, mean duration all = {ev_10['mean_duration_all']:.2f} vs uncensored = {ev_10['mean_duration_uncensored']:.2f} ngày.")
print(f"   - Tại tau = 0.1 mm: {int(ev_01['n_episodes'])} đợt mưa, {int(ev_01['n_right_censored'])} right-censored, mean duration all = {ev_01['mean_duration_all']:.2f} vs uncensored = {ev_01['mean_duration_uncensored']:.2f} ngày.")



🌧️ THỐNG KÊ ĐỢT MƯA (EPISODES) & BOUNDARY CENSORING (Observed duration within Dev Train 2001-2018):
 threshold_mm  wet_days  wet_pct  n_episodes  n_right_censored  mean_duration_all  mean_duration_uncensored  exact_threshold_days
          0.0      4600    72.68         403                 1              11.41                     11.42                  1729
          0.1      3857    60.94         466                 1               8.28                      8.28                    46
          0.5      3167    50.04         576                 1               5.50                      5.50                    10
          1.0      2747    43.40         654                 1               4.20                      4.20                     5
          2.0      2202    34.79         699                 1               3.15                      3.15                     5
          5.0      1348    21.30         643                 1               2.10                      2.10            

In [3]:
# 4. Trực quan hóa Độ nhạy Ngưỡng Mưa (Sensitivity Visualization)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 4.1. Occurrence Proportions (Rain vs No-Rain)
ax1 = axes[0, 0]
x_pos = np.arange(len(candidate_thresholds))
width = 0.35
ax1.bar(x_pos - width/2, threshold_comparison['no_rain_pct'], width, label=r'No-Rain ($Y \leq \tau$)', color='#2980b9', alpha=0.85)
ax1.bar(x_pos + width/2, threshold_comparison['rain_pct'], width, label=r'Rain ($Y > \tau$)', color='#27ae60', alpha=0.85)
for i, row in threshold_comparison.iterrows():
    ax1.text(i - width/2, row['no_rain_pct'] + 1, f"{row['no_rain_pct']:.1f}%", ha='center', fontsize=9)
    ax1.text(i + width/2, row['rain_pct'] + 1, f"{row['rain_pct']:.1f}%", ha='center', fontsize=9)
ax1.set_title(r'1. Tỷ lệ No-Rain ($Y \leq \tau$) vs Rain ($Y > \tau$)', fontsize=11, fontweight='bold')
ax1.set_xlabel(r'Ngưỡng Mưa $\tau$ (mm)')
ax1.set_ylabel('Tỷ lệ (%)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([str(th) for th in candidate_thresholds])
ax1.set_ylim(0, 95)
ax1.grid(True, linestyle='--', alpha=0.5, axis='y')
ax1.legend(loc='center right')

# 4.2. Incremental Data Mass between Thresholds (near-zero regime)
ax2 = axes[0, 1]
bracket_labels = [
    r'$[0, 0.1]$ mm' + '\n(near-zero)',
    r'$(0.1, 0.5]$ mm',
    r'$(0.5, 1.0]$ mm',
    r'$(1.0, 2.0]$ mm',
    r'$(2.0, 5.0]$ mm'
]
bracket_gaps = threshold_comparison['incremental_gap_pct'].iloc[1:].values
bars = ax2.bar(bracket_labels, bracket_gaps, color='#e67e22', edgecolor='black', alpha=0.85)
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.3, f"{yval:.2f}%", ha='center', fontsize=9, fontweight='bold')
ax2.set_title('2. Khối lượng Dữ liệu Dịch chuyển (Incremental Mass Brackets)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Khoảng Ngưỡng')
ax2.set_ylabel('Tỷ lệ phần trăm (% tổng số ngày)')
ax2.set_ylim(0, 18)
ax2.grid(True, linestyle='--', alpha=0.5, axis='y')

# 4.3. Conditional Rainfall Intensity on Rain-Days
ax3 = axes[1, 0]
ax3.plot(threshold_comparison['threshold_mm'], threshold_comparison['rain_mean_conditional'], marker='^', linewidth=2, color='#c0392b', label=r'Mean $E[Y \mid Y > \tau]$')
ax3.plot(threshold_comparison['threshold_mm'], threshold_comparison['rain_median_conditional'], marker='v', linewidth=2, color='#8e44ad', label=r'Median $\mathrm{Med}(Y \mid Y > \tau)$')
for _, row in threshold_comparison.iterrows():
    ax3.annotate(f"{row['rain_mean_conditional']:.1f}", (row['threshold_mm'], row['rain_mean_conditional'] + 0.4), fontsize=9, ha='center', color='#c0392b')
    ax3.annotate(f"{row['rain_median_conditional']:.1f}", (row['threshold_mm'], row['rain_median_conditional'] - 0.7), fontsize=9, ha='center', color='#8e44ad')
ax3.set_title(r'3. Cường độ Mưa có Điều kiện (Stage 2 Population: $Y > \tau$)', fontsize=11, fontweight='bold')
ax3.set_xlabel(r'Ngưỡng Mưa $\tau$ (mm)')
ax3.set_ylabel('Lượng mưa (mm)')
ax3.set_xticks(candidate_thresholds)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='upper left')

# 4.4. Exact-Threshold Days count #{Y == tau}
ax4 = axes[1, 1]
exact_bars = ax4.bar([str(th) for th in candidate_thresholds], threshold_comparison['exact_threshold_days'], color='#34495e', edgecolor='black', alpha=0.85)
for bar, cnt in zip(exact_bars, threshold_comparison['exact_threshold_days']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15, f'{cnt:,}', ha='center', fontsize=9)
ax4.set_title(r'4. Số Ngày Quan sát Đúng Bằng Ngưỡng $\#\{Y = \tau\}$', fontsize=11, fontweight='bold')
ax4.set_xlabel(r'Ngưỡng Mưa $\tau$ (mm)')
ax4.set_ylabel('Số ngày')
ax4.set_yscale('log')
ax4.set_ylim(0.5, 5000)
ax4.grid(True, linestyle='--', alpha=0.5, axis='y')

plt.tight_layout()
plt.show()


In [4]:
# 5. Xuất Artifact Báo cáo Threshold So sánh Thực nghiệm (JSON)
OUTPUT_DIR = ROOT_DIR / "reports" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ev_10 = df_events.loc[df_events['threshold_mm'] == 1.0].iloc[0]
ev_01 = df_events.loc[df_events['threshold_mm'] == 0.1].iloc[0]

near_zero_strict = int(((target > 0.0) & (target < 0.1)).sum())
near_zero_strict_pct = round(float(near_zero_strict / total_days) * 100.0, 2)
exact_01 = int((target == 0.1).sum())
near_zero_inclusive_pct = round(float(((target > 0.0) & (target <= 0.1)).sum() / total_days) * 100.0, 2)

threshold_artifact = {
    "architecture": {
        "main_forecasting_target": "Daily continuous rainfall Y_t (mm)",
        "target_dependence_on_tau": False,
        "roles_of_tau": [
            "EDA descriptive sensitivity analysis (wet/dry proportions, near-zero regime)",
            "Hurdle two-stage decomposition (Stage 1 occurrence & Stage 2 conditional amount)",
            "Optional secondary wet-day diagnostics (RainDay-MAE_tau)"
        ],
        "feature_engineering_policy": "No mandatory threshold-derived features in baseline F0-F3",
        "evaluation_metrics_policy": "Primary metrics (MAE, RMSE, CR-MAE@7) remain threshold-independent; RainDay-MAE is secondary diagnostic",
        "canonical_wet_day_definition": "tau = 1.0 mm/day (WMO RR1 standard: Wet if Y >= 1.0 mm, Dry if Y < 1.0 mm, unified across project)"
    },
    "metadata": {
        "data_split": "dev_train",
        "train_start": str(train_df['Ngày'].min().date()),
        "train_end": str(train_df['Ngày'].max().date()),
        "n_observations": len(train_df),
        "holdouts_untouched": "Validation (2018-05-01 to 2020-04-30, N=731) and Final Test (2020-05-01 to 2026-04-30, N=2191) strictly untouched."
    },
    "candidate_thresholds": candidate_thresholds,
    "comparison_table": threshold_comparison.to_dict(orient="records"),
    "episode_duration_robustness": df_events.to_dict(orient="records"),
    "boundary_censoring_note": (
        f"Episode duration statistics within Dev Train analysis window ({train_df['Ngày'].min().date()} to {train_df['Ngày'].max().date()}, N={len(train_df)}). "
        f"At tau=1.0 mm: {int(ev_10['n_episodes'])} episodes, {int(ev_10['n_right_censored'])} right-censored, "
        f"mean_duration_all = {ev_10['mean_duration_all']} days vs mean_duration_uncensored = {ev_10['mean_duration_uncensored']} days. "
        f"At tau=0.1 mm: {int(ev_01['n_episodes'])} episodes, {int(ev_01['n_right_censored'])} right-censored, "
        f"mean_duration_all = {ev_01['mean_duration_all']} days vs mean_duration_uncensored = {ev_01['mean_duration_uncensored']} days."
    ),
    "near_zero_precipitation_regime": {
        "bracket_0_to_0_1_mm_mass_pct": near_zero_strict_pct,
        "bracket_0_to_0_1_mm_days": near_zero_strict,
        "exact_0_1_mm_days": exact_01,
        "finding": (
            f"A significant proportion of observations ({near_zero_strict_pct}% strictly < 0.1 mm, {near_zero_inclusive_pct}% <= 0.1 mm) "
            "lies in the near-zero / very-light precipitation regime (0 < Y <= 0.1 mm), showing that "
            "wet-day definition substantially influences occurrence proportions without altering continuous rainfall values."
        )
    },
    "labeling_convention": {
        "dry": "Y < tau (or == 0.0 for tau = 0.0)",
        "wet": "Y >= tau (or > 0.0 for tau = 0.0)"
    },
    "canonical_project_threshold_mm": 1.0,
    "initial_project_candidate": 0.1,
    "exact_threshold_counts": {str(th): int((target == th).sum()) for th in candidate_thresholds},
    "protocol_guidance": (
        "Candidate rainfall thresholds are compared empirically to characterize rainfall occurrence, "
        "class balance, and conditional intensity. Unified canonical wet-day convention is Y >= 1.0 mm. "
        "The main forecasting target remains continuous daily rainfall Y_t (mm)."
    )
}

artifact_path = OUTPUT_DIR / "01.1_threshold_report.json"
with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(threshold_artifact, f, indent=2, ensure_ascii=False)

print(f"💾 Threshold comparison artifact exported to: {artifact_path}")


💾 Threshold comparison artifact exported to: D:\DS108_project\reports\eda\01.1_threshold_report.json
